# Fine-tune Qwen2.5-Coder-1.5B on LC-QuAD 1.0

Produces the checkpoint that `type_constrained_generation.py` loads from
`model/qwen25-coder-1.5b-lcquad`.

The model is trained as a plain causal LM on `Question: ...\nSPARQL:\n<query>`,
which is the exact prompt the generator builds at inference time. Targets go
through `test.py`'s `canonicalize` first, so the model is taught the one surface
spelling the constrained decoder can actually emit (single spaces, `COUNT(?uri)`,
`{ ` / ` }`, ` . ` between triples, no dot before the closing brace).

This is a **full fine-tune** -- every weight is trained, there is no adapter, and
the only dependencies are torch, transformers and accelerate. The best checkpoint,
selected by exact match on a held-out validation split (the same metric `test.py`
reports), is saved straight to `model.safetensors` with no merge step.

## 1. Environment

In [ ]:
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> A100 (or T4)."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()  # A100 yes, T4 no
print(f"torch {torch.__version__}  |  {GPU_NAME} ({GPU_GB:.0f} GB)  |  bf16 supported: {BF16}")

# A Colab VM whose torch/torchvision pairing has been broken by an upgrade fails
# later inside transformers' image utils with a misleading import error. Check the
# op directly rather than comparing version strings.
try:
    torch.ops.torchvision.nms
    print("torchvision ops OK")
except Exception as e:
    print(f"WARNING: torchvision is not loadable against this torch ({e}).\n"
          "pip cannot repair this -- use Runtime -> Disconnect and delete runtime, "
          "then start over without upgrading torch.")

In [ ]:
# Install only what is genuinely absent, and never --upgrade: lifting torch out from
# under Colab's preinstalled torchvision breaks every transformers import afterwards.
import importlib.util
import sys

REQUIRED = ["transformers", "accelerate", "safetensors"]
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies already present")

import transformers

print(f"transformers {transformers.__version__}")

## 2. Download LC-QuAD 1.0

In [ ]:
import urllib.request
from pathlib import Path

# LC-QuAD v1 lives on the `data` branch of the AskNowQA repo: 4000 train / 1000 test
# question-SPARQL pairs, all of them DBpedia.
LCQUAD_URLS = {
    "train-data.json": "https://raw.githubusercontent.com/AskNowQA/LC-QuAD/data/train-data.json",
    "test-data.json": "https://raw.githubusercontent.com/AskNowQA/LC-QuAD/data/test-data.json",
}
DATA_DIR = Path("/content/lcquad_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

for name, url in LCQUAD_URLS.items():
    dest = DATA_DIR / name
    if dest.exists():
        print(f"{name} already downloaded ({dest.stat().st_size:,} bytes)")
        continue
    urllib.request.urlretrieve(url, dest)
    print(f"{name}: {dest.stat().st_size:,} bytes")

In [ ]:
import json

raw_train = json.loads((DATA_DIR / "train-data.json").read_text(encoding="utf-8"))
print(f"{len(raw_train)} train records")
print(json.dumps(raw_train[0], indent=2))


## 3. Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"

PROMPT = "Question: {question}\nSPARQL:\n"

OUT_ROOT = Path("/content/qwen_lcquad")
SAVE_DIR = OUT_ROOT / "qwen25-coder-1.5b-lcquad"  # best checkpoint, model.safetensors
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MAX_LEN = 200

VAL_SIZE = 400   # held out from the 4000 train records
SEED = 0
EPOCHS = 10
LEARNING_RATE = 1e-5  # full fine-tune

if BF16:
    # A100
    BATCH_SIZE, GRAD_ACCUM, GRAD_CKPT, OPTIM = 16, 2, False, "adamw_torch_fused"
else:
    # T4
    BATCH_SIZE, GRAD_ACCUM, GRAD_CKPT, OPTIM = 4, 8, True, "adafactor"

EVAL_BATCH_SIZE = 32   # batched greedy generation for the exact-match metric
MAX_NEW_TOKENS = 160

print(f"{GPU_NAME}: batch {BATCH_SIZE} x accum {GRAD_ACCUM} "
      f"(effective {BATCH_SIZE * GRAD_ACCUM}), {'bf16' if BF16 else 'fp16'}, "
      f"{OPTIM}, gradient checkpointing {GRAD_CKPT}")
if not BF16:
    print("\nNOTE: full fine-tuning a 1.5B model on a T4 leaves very little headroom.\n"
          "If this OOMs, halve BATCH_SIZE and double GRAD_ACCUM.")

## 4. Build the training pairs

`canonicalize` is copied verbatim from `test.py`. Training on the canonical form
matters: the gold queries carry doubled spaces, `COUNT( ?uri )`, and a trailing
` . ` before `}` that the beginning-template and relation grammars can never
produce, so a model trained on the raw strings would be fighting the decoder.

Records whose canonical form does not open with one of the five legal beginning
templates are dropped -- there is exactly one such row in LC-QuAD train (a
truncated query that starts `uri WHERE {`).

In [ ]:
import re


def canonicalize(q):
    # _scratch_trie.canonicalize plus one extra step: drop a trailing dot
    # before the closing brace. Legal SPARQL and present in ~28% of gold
    # queries, but the generator's grammar can never emit it, so without
    # this those queries could never exact-match.
    q = " ".join(q.split())
    q = q.replace("COUNT( ?uri )", "COUNT(?uri)")
    q = q.replace("{", "{ ").replace("}", " }")
    q = re.sub(r"\s*\.\s*(?![^<]*>)", " . ", q)
    return " ".join(q.split()).replace(" . }", " }")


# the five BEGINNING_TEMPLATE alternatives, spelled out over var ::= "?uri" | "?x"
LEGAL_HEADS = (
    "SELECT DISTINCT ?uri WHERE { ?uri",
    "SELECT DISTINCT ?uri WHERE { ?x",
    "SELECT DISTINCT ?uri WHERE { <",
    "SELECT DISTINCT COUNT(?uri) WHERE { ?uri",
    "SELECT DISTINCT COUNT(?uri) WHERE { ?x",
    "SELECT DISTINCT COUNT(?uri) WHERE { <",
    "ASK WHERE { <",
)


def build_pairs(records):
    """(question, canonical query) pairs, minus anything off-grammar."""
    pairs, dropped = [], []
    for r in records:
        target = canonicalize(r["sparql_query"])
        if not target.startswith(LEGAL_HEADS):
            dropped.append(target)
            continue
        pairs.append({"question": r["corrected_question"], "target": target})
    return pairs, dropped


train_pairs, train_dropped = build_pairs(raw_train)
print(f"train: {len(train_pairs)} usable, {len(train_dropped)} dropped")
for d in train_dropped:
    print("  dropped:", d[:110])

print("\nexample pair:")
print(repr(PROMPT.format(question=train_pairs[0]["question"])))
print(repr(train_pairs[0]["target"]))

In [ ]:
import random

# held-out validation split for model selection; the LC-QuAD test split stays
# untouched so test.py stays an honest measurement
shuffled = train_pairs[:]
random.Random(SEED).shuffle(shuffled)
val_pairs = shuffled[:VAL_SIZE]
fit_pairs = shuffled[VAL_SIZE:]
print(f"{len(fit_pairs)} training / {len(val_pairs)} validation")

## 5. Tokenize

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD_ID = tokenizer.pad_token_id
EOS_ID = tokenizer.eos_token_id


class SPARQLDataset(Dataset):
    """Prompt+target as one causal-LM sequence, loss masked to the target.

    add_special_tokens=False everywhere, matching how the generator tokenizes its
    prompt; EOS is appended to the target so unconstrained generation knows to stop
    (the constrained decoder terminates on the grammar instead and ignores it).
    """

    def __init__(self, pairs):
        self.rows = []
        for p in pairs:
            prompt_ids = tokenizer(PROMPT.format(question=p["question"]),
                                   add_special_tokens=False).input_ids
            target_ids = tokenizer(p["target"], add_special_tokens=False).input_ids + [EOS_ID]
            ids = prompt_ids + target_ids
            labels = [-100] * len(prompt_ids) + target_ids
            ids, labels = ids[:MAX_LEN], labels[:MAX_LEN]
            self.rows.append((ids, labels))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        ids, labels = self.rows[i]
        return {"input_ids": ids, "labels": labels}


def collate(batch):
    """Right-pad to the longest sequence in the batch; -100 keeps pads out of the loss."""
    width = max(len(b["input_ids"]) for b in batch)
    input_ids, labels, attention_mask = [], [], []
    for b in batch:
        pad = width - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [PAD_ID] * pad)
        labels.append(b["labels"] + [-100] * pad)
        attention_mask.append([1] * len(b["input_ids"]) + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


train_ds = SPARQLDataset(fit_pairs)
val_ds = SPARQLDataset(val_pairs)
print(f"{len(train_ds)} train / {len(val_ds)} val sequences")

## 6. Load the model

Weights are loaded in fp32 and the matmuls run in bf16 via `bf16=True` in the
trainer -- the standard mixed-precision setup, where the optimizer keeps fp32
master weights so small updates are not lost to rounding. Every parameter is
trainable; nothing is frozen and there is no adapter to merge later.

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.config.pad_token_id = PAD_ID
model.config.use_cache = False  # clashes with gradient checkpointing; re-enabled to generate

if GRAD_CKPT:
    model.gradient_checkpointing_enable()

model.cuda()
n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{n_params / 1e9:.2f}B parameters, {n_train / 1e9:.2f}B trainable "
      f"({n_train / n_params:.0%})")

## 7. Exact match on the validation split

Selecting on validation loss would pick the checkpoint that assigns the best
probability to gold tokens, which is not the same thing as writing the right
query. This generates greedily and compares canonical strings -- the metric
`test.py` reports, minus the grammar constraints, so it reads as a lower bound on
what the constrained decoder will do.

In [ ]:
# generation pads on the left so every sequence in a batch ends at its last real token
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token


@torch.no_grad()
def exact_match(pairs, batch_size=EVAL_BATCH_SIZE, show=0):
    """Fraction of `pairs` whose greedy continuation canonicalizes to the gold query."""
    was_training, was_cached = model.training, model.config.use_cache
    model.eval()
    model.config.use_cache = True

    hits, samples = 0, []
    for start in range(0, len(pairs), batch_size):
        chunk = pairs[start:start + batch_size]
        enc = gen_tokenizer([PROMPT.format(question=p["question"]) for p in chunk],
                            return_tensors="pt", padding=True,
                            add_special_tokens=False).to(model.device)
        with torch.autocast("cuda", dtype=torch.bfloat16 if BF16 else torch.float16):
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 num_beams=1, pad_token_id=PAD_ID, eos_token_id=EOS_ID)
        completions = gen_tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                                 skip_special_tokens=True)
        for pair, text in zip(chunk, completions):
            produced = canonicalize(text)
            hit = produced == pair["target"]
            hits += hit
            if len(samples) < show:
                samples.append((pair["question"], produced, pair["target"], hit))

    model.config.use_cache = was_cached
    model.train(was_training)

    for question, produced, gold, hit in samples:
        print(f"  [{'HIT ' if hit else 'MISS'}] {question}")
        print(f"         got  {produced}")
        if not hit:
            print(f"         gold {gold}")
    return hits / len(pairs)

In [ ]:
from transformers import TrainerCallback


class BestExactMatch(TrainerCallback):
    """Score validation EM at each epoch end and keep the best weights in CPU RAM.

    The copy is taken in bf16: 3.1 GB rather than the 6.2 GB the fp32 training
    weights occupy, and bf16 is the dtype the checkpoint is saved in anyway.
    Keeping it in RAM avoids writing a 3 GB checkpoint on every improvement.
    """

    def __init__(self):
        self.best = -1.0
        self.best_epoch = None
        self.best_state = None
        self.history = []

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = round(state.epoch)
        em = exact_match(val_pairs, show=3 if epoch == 1 else 0)
        self.history.append((epoch, em))
        flag = ""
        if em > self.best:
            self.best, self.best_epoch = em, epoch
            self.best_state = {k: v.detach().to("cpu", torch.bfloat16)
                               for k, v in kwargs["model"].state_dict().items()}
            flag = "  <- best so far, weights kept"
        print(f"epoch {epoch}: validation exact match {em:.2%}{flag}", flush=True)


best_tracker = BestExactMatch()

## 8. Train

In [ ]:
import inspect

from transformers import Trainer, TrainingArguments

targs = dict(
    output_dir=str(OUT_ROOT / "trainer"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=GRAD_CKPT,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=1.0,
    weight_decay=0.0,
    optim=OPTIM,
    bf16=BF16,
    fp16=not BF16,
    logging_steps=25,
    save_strategy="no",      # BestExactMatch owns checkpointing
    report_to="none",
    seed=SEED,
)
# transformers renamed evaluation_strategy -> eval_strategy; support both
accepted = set(inspect.signature(TrainingArguments.__init__).parameters)
targs["eval_strategy" if "eval_strategy" in accepted else "evaluation_strategy"] = "epoch"
dropped = [k for k in targs if k not in accepted]
for k in dropped:
    targs.pop(k)
if dropped:
    print("unsupported TrainingArguments dropped for this version:", dropped)

trainer = Trainer(
    model=model,
    args=TrainingArguments(**targs),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate,
    callbacks=[best_tracker],
)
trainer.train()

In [ ]:
print("validation exact match by epoch")
for epoch, em in best_tracker.history:
    print(f"  epoch {epoch}: {em:.2%}")
print(f"\nbest: epoch {best_tracker.best_epoch} at {best_tracker.best:.2%}")
assert best_tracker.best_state is not None, "no weights were kept"

## 9. Save the best checkpoint as safetensors

The optimizer state is dropped first to free the GPU, then the best weights are
loaded back and written in bf16 -- the dtype `type_constrained_generation.py`
loads with on CUDA (`torch_dtype=torch.bfloat16`). At 1.5B parameters this is a
single ~3.1 GB `model.safetensors`, under the shard threshold.

In [ ]:
import gc

del trainer
gc.collect()
torch.cuda.empty_cache()

model = model.to(torch.bfloat16)
model.load_state_dict(best_tracker.best_state)
model.config.use_cache = True
model.save_pretrained(str(SAVE_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(SAVE_DIR))

best_tracker.best_state = None
del model
gc.collect()
torch.cuda.empty_cache()

for f in sorted(SAVE_DIR.iterdir()):
    print(f"  {f.name:<32} {f.stat().st_size / 1e6:>9,.1f} MB")

## 10. Smoke-test the saved checkpoint

In [ ]:
check_model = AutoModelForCausalLM.from_pretrained(
    SAVE_DIR, torch_dtype=torch.bfloat16 if BF16 else torch.float16).cuda().eval()
check_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR, padding_side="left")

questions = [
    "How many movies did Stanley Kubrick direct?",
    "Name the municipality of Roberto Clemente Bridge ?",
    "Was Kobe Bryant a basketball player?",
]
enc = check_tokenizer([PROMPT.format(question=q) for q in questions],
                      return_tensors="pt", padding=True,
                      add_special_tokens=False).to(check_model.device)
with torch.no_grad():
    out = check_model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                               num_beams=1, pad_token_id=EOS_ID, eos_token_id=EOS_ID)
for q, text in zip(questions, check_tokenizer.batch_decode(
        out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
    print(f"{q}\n  {canonicalize(text)}\n")

del check_model
gc.collect()
torch.cuda.empty_cache()

## 11. Get the checkpoint off Colab

Copy the directory to Drive, then drop it into the repo at
`model/qwen25-coder-1.5b-lcquad` (the default `MODEL_PATH`, overridable in `.env`).
Set `COPY_TO_DRIVE = False` to skip and download `SAVE_DIR` by hand instead.

In [ ]:
COPY_TO_DRIVE = True
DRIVE_DEST = "/content/drive/MyDrive/qwen25-coder-1.5b-lcquad"

if COPY_TO_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    if Path(DRIVE_DEST).exists():
        shutil.rmtree(DRIVE_DEST)
    shutil.copytree(SAVE_DIR, DRIVE_DEST)
    total = sum(f.stat().st_size for f in Path(DRIVE_DEST).iterdir())
    print(f"copied {total / 1e9:.2f} GB to {DRIVE_DEST}")
else:
    print(f"checkpoint left at {SAVE_DIR}")

print("\nPlace the directory at model/qwen25-coder-1.5b-lcquad in the repo, then run:")
print("  python type_constrained_generation.py     # single-question smoke test")
print("  python test.py                            # LC-QuAD test-split evaluation")